# BAA10Y — conventional-methods backtest comparison

| Spec | Begin date | End date | Stride |
|---|---|---|---|
| `baa10y_smoke` | 2025-10-06 | 2025-11-14 | 5 (weekly origins, ~6) |
| `baa10y_backtest_2025` | 2025-01-06 | 2025-12-22 | 5 (weekly origins, ~50) |
| `baa10y_stress_2020` | 2020-02-03 | 2020-04-30 | 1 (daily origins across the COVID crash) |
| `baa10y_eval_2026` | 2026-02-03 | 2026-03-24 | 5 (weekly origins, 8 total; protected, `max_runs: 5`) |

---
## 1. Setup

The heavy lifting lives in helper modules alongside this notebook:

- `data.py`        builds the `DataService` (return targets + the covariate panel).
- `predictors/`    the BAA10Y LLM-Process prompt and sampling recipe.
- `leaderboard.py` turns cached results into the `RESULTS_DF` leaderboard frame.
- `analysis.py` / `plots.py`  direction metrics, styled tables, and figures.

We build **one** data service that registers the three return targets plus the
full covariate panel. Target-only predictors simply ignore the registered
covariates; the covariate variants read them.

**First run on the 2025/2026 windows?** Warm the caches to the present first:
`uv run python scripts/fetch_sp500_market.py --refresh` (Yahoo)
and `uv run python scripts/fetch_fred.py` (traget and macro covariates).

In [1]:
from __future__ import annotations

import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import yaml
from dotenv import load_dotenv


warnings.filterwarnings("ignore")


# Resolve the repo root robustly (walk up for the workspace markers) so the
# proxy creds load regardless of the kernel's working directory.
def _repo_root() -> Path:
    here = Path.cwd().resolve()
    for cand in (here, *here.parents):
        if (cand / "pyproject.toml").exists() and (cand / "aieng-forecasting").is_dir():
            return cand
    return here


ROOT = _repo_root()
load_dotenv(ROOT / ".env", override=False)  # LLMP rows call the Vector proxy — need PROXY_* set


from aieng.forecasting.evaluation import (
    MultiTargetBacktestSpec,
    MultiTargetEvalSpec,
    cached_multi_backtest,
    describe_spec,
    multi_evaluate,
)
from aieng.forecasting.methods import (
    DartsAutoARIMAPredictor,
    DartsExponentialSmoothingPredictor,
    DartsKalmanForecasterPredictor,
    DartsLightGBMPredictor,
    DartsLinearRegressionPredictor,
    LastValuePredictor,
)


In [2]:
# BAA10Y imports

from BAA10Y_forecasting import (
    DEFAULT_COVARIATE_SERIES_IDS,
    HYOAS_OPTIONAL_COVARIATE_SERIES_IDS,
    build_baa10y_multivariate_service,
)
from BAA10Y_forecasting.leaderboard import build_leaderboard, build_spread_change_compare_frame

from BAA10Y_forecasting.plots import (
    display_multivariate_backtest_leaderboard,
    plotly_display_multivariate_backtest_leaderboard,
    plot_spread_change_forecast_vs_actual_multi,
    plotly_spread_change_forecast_vs_actual_multi,
    plot_baa10y_spread_change_recent,
    plotly_baa10y_spread_change_recent
)
from BAA10Y_forecasting.predictors import build_baa10y_llmp_sampled_trajectory

In [3]:
#

SPECS_DIR = ROOT / "implementations" / "BAA10Y_forecasting" / "specs"

PREDICTIONS_DIR = ROOT / "data" / "predictions"

In [4]:
# Data environment (not part of any single experiment): how far back to load
# price/covariate history, and whether to re-fetch caches from source.
DATA_HISTORY_START = "2016-01-01"
REFRESH_CACHE = False

#

svc = build_baa10y_multivariate_service(
    windows=(1, 5, 21),
    include_covariates=True,
    covariate_series_ids=DEFAULT_COVARIATE_SERIES_IDS,
    start=DATA_HISTORY_START,
    refresh=REFRESH_CACHE,
)

# The covariate panel available to the with-covariates predictors below (an
# optional FRED feed may be skipped, so filter to what actually registered).
registered = set(svc.series_ids)
COVARIATES = [c for c in DEFAULT_COVARIATE_SERIES_IDS if c in registered]
print("targets:", sorted(s for s in registered if s.startswith("baa10y_change")))
print(f"covariates registered: {len(COVARIATES)} / {len(DEFAULT_COVARIATE_SERIES_IDS)} requested")

targets: ['baa10y_change_1b', 'baa10y_change_21b', 'baa10y_change_5b']
covariates registered: 10 / 11 requested


---
## 2. Configuration

One selector picks the backtest window; everything downstream adapts. Specs are
the source of truth for the **window and tasks** — not the predictors.

In [5]:
# ── Experiment configuration ──────────────────────────────────────────────────
# EXPERIMENT_CONFIG chooses which window drives the backtest (Section 5); all
# downstream cells adapt. The protected 2026 eval (Section 7) is fixed.
#
#   "smoke"          short late-2025 window (~6 weekly origins), post-cutoff.
#                    Fast default; LLM-Process predictors ON.
#   "backtest_2025"  weekly origins across all of 2025 (~50), post-cutoff.
#                    The full comparison; LLMP is slow here — trim the roster.
#   "stress_2020"    the COVID crash (daily origins Feb–Apr 2020). NUMERICAL
#                    ONLY — pre-cutoff is leaked for LLMs, so the predictors cell
#                    drops the LLMP rows for this config.

#EXPERIMENT_CONFIG = "smoke"
#EXPERIMENT_CONFIG = "stress_2020"
EXPERIMENT_CONFIG = "backtest_2025"

_BACKTEST_SPEC_FILES = {
    "smoke": "baa10y_smoke.yaml",
    "backtest_2025": "baa10y_backtest_2025.yaml",
    "stress_2020": "baa10y_stress_2020.yaml",
}
_EVAL_SPEC_FILE = "baa10y_eval_2026.yaml"

# 2020 is pre-cutoff → keep the LLM-Process out of the comparison (see Section 0).
POST_CUTOFF = EXPERIMENT_CONFIG in {"smoke", "backtest_2025"}

with (SPECS_DIR / _BACKTEST_SPEC_FILES[EXPERIMENT_CONFIG]).open() as f:
    backtest_spec = MultiTargetBacktestSpec.model_validate(yaml.safe_load(f))

print(
    f"Config: {EXPERIMENT_CONFIG!r}  →  {_BACKTEST_SPEC_FILES[EXPERIMENT_CONFIG]}  "
    f"(LLMP {'on' if POST_CUTOFF else 'OFF — pre-cutoff'})"
)
#print(describe_spec(backtest_spec, data_service=svc))

Config: 'backtest_2025'  →  baa10y_backtest_2025.yaml  (LLMP on)


---
## 4. Predictors — configured in code

This is where you choose the roster. Each predictor implements the same
`Predictor` API against the loaded spec; the conventional methods come straight
from `aieng.forecasting.methods`, and the LLM-Process variants come from the
BAA10Y recipe in `predictors/`. The covariate variants read the registered
`COVARIATES` panel; the target-only variants pass `covariate_series_ids=None`.

| Group | Predictor | Covariates? |
|---|---|---|
| Naive floor | `LastValuePredictor` | — |
| Classical | `DartsExponentialSmoothingPredictor` (ETS), `DartsKalmanForecasterPredictor` | — |
| ML regression | `DartsLinearRegressionPredictor`, `DartsLightGBMPredictor` | target-only **and** + panel |
| LLM-Process | `build_sp500_llmp_sampled_trajectory` | target-only **and** + panel (post-cutoff only) |

`AutoARIMA` is left commented (accurate but slow); add it back for a classical
sweep. The `LLMP (target)` vs `LLMP + cov` pair is the centerpiece — its CRPS gap
answers whether an LLM can use the same exogenous panel the ML methods do.

In [6]:
# Shared hyperparameters for the Darts regression models.
LAGS = 5  # autoregressive lags (and past-covariate lags) for the regression models
NUM_SAMPLES = 100  # empirical-quantile sample count for the probabilistic Darts models
LGBM_KWARGS = {"num_threads": 3, "n_jobs": 1, "verbosity": -1, "max_depth": 4, "num_leaves": 16}

# ── Naive floor + classical (univariate) ──────────────────────────────────────
naive = LastValuePredictor()
ets = DartsExponentialSmoothingPredictor(num_samples=NUM_SAMPLES)
kalman = DartsKalmanForecasterPredictor(num_samples=NUM_SAMPLES)
autoarima = DartsAutoARIMAPredictor(num_samples=NUM_SAMPLES)  # accurate but the slowest classical method

# ── ML regression — target-only vs the covariate panel ────────────────────────
linreg = DartsLinearRegressionPredictor(lags=LAGS, covariate_series_ids=None, num_samples=NUM_SAMPLES)
linreg_cov = DartsLinearRegressionPredictor(
    lags=LAGS, lags_past_covariates=LAGS, covariate_series_ids=COVARIATES, num_samples=NUM_SAMPLES
)
lightgbm = DartsLightGBMPredictor(
    lags=LAGS, covariate_series_ids=None, num_samples=NUM_SAMPLES, lgbm_kwargs=LGBM_KWARGS
)
lightgbm_cov = DartsLightGBMPredictor(
    lags=LAGS,
    lags_past_covariates=LAGS,
    covariate_series_ids=COVARIATES,
    num_samples=NUM_SAMPLES,
    lgbm_kwargs={**LGBM_KWARGS}
)

#all_predictors = [naive, ets, kalman, autoarima, linreg, linreg_cov, lightgbm, lightgbm_cov]
all_predictors = [naive, linreg, lightgbm]

# Which covariate panel each predictor consumes — drives the leaderboard's
# covariate columns. Predictors absent from this map are treated as target-only.
PREDICTOR_COVARIATES = {
    linreg_cov.predictor_id: COVARIATES,
    lightgbm_cov.predictor_id: COVARIATES,
}
# Short labels for the leaderboard "model" column and the CRPS charts.
PREDICTOR_LABELS = {
    naive.predictor_id: "Naive",
    ets.predictor_id: "ETS",
    kalman.predictor_id: "Kalman",
    autoarima.predictor_id: "AutoARIMA",
    linreg.predictor_id: "LinReg",
    linreg_cov.predictor_id: "LinReg + cov",
    lightgbm.predictor_id: "LightGBM",
    lightgbm_cov.predictor_id: "LightGBM + cov",
}

# ── LLM-Process (sampled trajectories) — target-only vs with-covariates ────────
# Gated on POST_CUTOFF: the 2020 stress window is pre-cutoff and leaked for LLMs,
# so we keep the LLMP rows out of that comparison entirely. The recipe (prompt
# framing, defaults) lives in predictors/llmp_sampled_trajectory.py; tune it
# there, or override model= / n_samples= / history_window= per call here.
if POST_CUTOFF:
    llmp = build_baa10y_llmp_sampled_trajectory(n_samples=8, history_window=48)
    llmp_cov = build_baa10y_llmp_sampled_trajectory(n_samples=8, history_window=48, covariate_series_ids=COVARIATES)
    all_predictors += [llmp, llmp_cov]
    PREDICTOR_COVARIATES[llmp_cov.predictor_id] = COVARIATES
    PREDICTOR_LABELS[llmp.predictor_id] = "LLMP (target)"
    PREDICTOR_LABELS[llmp_cov.predictor_id] = "LLMP + cov"

print(f"{len(all_predictors)} predictors configured (LLMP {'on' if POST_CUTOFF else 'off'}):")
for p in all_predictors:
    print(f"  {p.predictor_id}")

5 predictors configured (LLMP on):
  last_value_naive
  darts_linreg
  darts_lightgbm
  llmp_sampled_trajectories_baa10y_v1_target_h48_n8[gemini-3.1-flash-lite-preview]
  llmp_sampled_trajectories_baa10y_v1_cov_h48_n8[gemini-3.1-flash-lite-preview]


# NEW

In [7]:
def run_experiment(EXPERIMENT_CONFIG: str, all_predictors, force_refresh=True)-> pd.DataFrame:

    with (SPECS_DIR / _BACKTEST_SPEC_FILES[EXPERIMENT_CONFIG]).open() as f:
        backtest_spec = MultiTargetBacktestSpec.model_validate(yaml.safe_load(f))

    #

    results_by_predictor: dict[str, dict[str, object]] = {}

    for predictor in all_predictors:
        print(f"Running {predictor.predictor_id} ...", flush=True)
        results_by_predictor[predictor.predictor_id] = cached_multi_backtest(
            predictor=predictor,
            spec=backtest_spec,
            data_service=svc,
            store_dir=PREDICTIONS_DIR,
            force_refresh=force_refresh
        )
        for task_id, result in results_by_predictor[predictor.predictor_id].items():
            print(f"  {task_id:18s}  mean CRPS = {result.mean_score:.5f}  ({len(result.predictions)} preds)")

    RESULTS_DF = build_leaderboard(
        results_by_predictor,
        svc,
        covariates_by_predictor=PREDICTOR_COVARIATES,
        labels_by_predictor=PREDICTOR_LABELS,
    )

    return RESULTS_DF

In [8]:
from scipy.stats import binomtest

def _worse_than_random(row: pd.Series) -> float:
    n_correct = round(row["dir_accuracy"] * row["dir_n_eval"])
    return binomtest(int(n_correct), int(row["dir_n_eval"]), p=0.5, alternative="less").pvalue

def check_direction_accuracy(results_df: pd.DataFrame) -> bool:

    dir_rows = results_df.dropna(subset=["dir_accuracy"])
    dir_rows = dir_rows[dir_rows["dir_n_eval"] > 0]

    pvals = dir_rows.apply(_worse_than_random, axis=1)
    flagged = dir_rows.loc[pvals < 0.1, ["model", "horizon", "dir_accuracy", "dir_n_eval"]].assign(p_value=pvals[pvals < 0.1])

    print("Sanity Check — direction accuracy vs. 0.5 baseline (one-sided binomial p < 0.1):")
    print(
        dir_rows[["model", "horizon", "dir_accuracy", "dir_n_eval"]]
        .sort_values(["horizon", "dir_accuracy"])
        .to_string(index=False, float_format=lambda v: f"{v:.4f}")
    )

    if flagged.empty:
        print("\nPASS — no model is significantly worse than a coin flip on direction.")
    else:
        print(f"\nFLAGGED — {len(flagged)} (model, horizon) row(s) directionally worse than random:")
        print(flagged.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

# FIXME: return PASS/FLAGGED, flagged_reasons
    return flagged.empty

# Smoke Test

In [9]:
smoke_results_df = run_experiment("smoke", [naive, linreg, lightgbm])

Running last_value_naive ...
  baa10y_change_1b    mean CRPS = 2.00000  (6 preds)
  baa10y_change_5b    mean CRPS = 6.50000  (6 preds)
  baa10y_change_21b   mean CRPS = 8.50000  (6 preds)
Running darts_linreg ...
  baa10y_change_1b    mean CRPS = 0.92976  (6 preds)
  baa10y_change_5b    mean CRPS = 3.35111  (6 preds)
  baa10y_change_21b   mean CRPS = 7.00788  (6 preds)
Running darts_lightgbm ...
  baa10y_change_1b    mean CRPS = 1.00644  (6 preds)
  baa10y_change_5b    mean CRPS = 3.33131  (6 preds)
  baa10y_change_21b   mean CRPS = 6.41963  (6 preds)


In [10]:
check_direction_accuracy(smoke_results_df)

Sanity Check — direction accuracy vs. 0.5 baseline (one-sided binomial p < 0.1):
   model  horizon  dir_accuracy  dir_n_eval
LightGBM        1        0.3333           6
  LinReg        1        0.5000           6
   Naive        1        0.6667           6
LightGBM        5        0.5000           6
  LinReg        5        0.5000           6
   Naive        5        0.6667           6
  LinReg       21        0.1667           6
   Naive       21        0.3333           6
LightGBM       21        0.5000           6

PASS — no model is significantly worse than a coin flip on direction.


True

In [11]:
plotly_display_multivariate_backtest_leaderboard(smoke_results_df)

,horizon,target,model,uses_covariates,n_covariates,covariates,predictor_id,mean_crps,n_scores,n_predictions,skipped_origins,dir_precision_up,dir_recall_up,dir_f1_up,dir_accuracy,dir_roc_auc_prob_up,dir_n_eval
0,1,baa10y_change_1b,LinReg,False,0,—,darts_linreg,0.92976,6,6,0,0.000,0.000,0.000,0.500,0.667,6
1,1,baa10y_change_1b,LightGBM,False,0,—,darts_lightgbm,1.00644,6,6,0,0.400,0.667,0.500,0.333,0.222,6
2,1,baa10y_change_1b,Naive,False,0,—,last_value_naive,2.00000,6,6,0,1.000,0.333,0.500,0.667,0.556,6
3,5,baa10y_change_5b,LightGBM,False,0,—,darts_lightgbm,3.33131,6,6,0,1.000,0.250,0.400,0.500,0.625,6
4,5,baa10y_change_5b,LinReg,False,0,—,darts_linreg,3.35111,6,6,0,1.000,0.250,0.400,0.500,0.625,6
5,5,baa10y_change_5b,Naive,False,0,—,last_value_naive,6.50000,6,6,0,0.750,0.750,0.750,0.667,0.625,6
6,21,baa10y_change_21b,LightGBM,False,0,—,darts_lightgbm,6.41963,6,6,0,1.000,0.400,0.571,0.500,0.400,6
7,21,baa10y_change_21b,LinReg,False,0,—,darts_linreg,7.00788,6,6,0,0.000,0.000,0.000,0.167,0.200,6
8,21,baa10y_change_21b,Naive,False,0,—,last_value_naive,8.50000,6,6,0,0.667,0.400,0.500,0.333,0.200,6


# 2025 Backtest

In [9]:
backtest_2025_results_df = run_experiment("backtest_2025", [naive, linreg, lightgbm])

Running last_value_naive ...
  baa10y_change_1b    mean CRPS = 2.50980  (51 preds)
  baa10y_change_5b    mean CRPS = 6.80392  (51 preds)
  baa10y_change_21b   mean CRPS = 10.07843  (51 preds)
Running darts_linreg ...
  baa10y_change_1b    mean CRPS = 1.15341  (51 preds)
  baa10y_change_5b    mean CRPS = 3.39920  (51 preds)
  baa10y_change_21b   mean CRPS = 6.95295  (51 preds)
Running darts_lightgbm ...
  baa10y_change_1b    mean CRPS = 1.22740  (51 preds)
  baa10y_change_5b    mean CRPS = 3.45349  (51 preds)
  baa10y_change_21b   mean CRPS = 6.66198  (51 preds)


In [10]:
check_direction_accuracy(backtest_2025_results_df)

Sanity Check — direction accuracy vs. 0.5 baseline (one-sided binomial p < 0.1):
   model  horizon  dir_accuracy  dir_n_eval
   Naive        1        0.5294          51
LightGBM        1        0.5490          51
  LinReg        1        0.5882          51
   Naive        5        0.4314          51
  LinReg        5        0.4902          51
LightGBM        5        0.4902          51
  LinReg       21        0.5098          51
LightGBM       21        0.5686          51
   Naive       21        0.5686          51

PASS — no model is significantly worse than a coin flip on direction.


True

In [11]:
plotly_display_multivariate_backtest_leaderboard(backtest_2025_results_df)


,horizon,target,model,uses_covariates,n_covariates,covariates,predictor_id,mean_crps,n_scores,n_predictions,skipped_origins,dir_precision_up,dir_recall_up,dir_f1_up,dir_accuracy,dir_roc_auc_prob_up,dir_n_eval
0,1,baa10y_change_1b,LinReg,False,0,—,darts_linreg,1.15341,51,51,0,0.000,0.000,0.000,0.588,0.605,51
1,1,baa10y_change_1b,LightGBM,False,0,—,darts_lightgbm,1.22740,51,51,0,0.450,0.429,0.439,0.549,0.448,51
2,1,baa10y_change_1b,Naive,False,0,—,last_value_naive,2.50980,51,51,0,0.412,0.333,0.368,0.529,0.479,51
3,5,baa10y_change_5b,LinReg,False,0,—,darts_linreg,3.39920,51,51,0,0.400,0.080,0.133,0.490,0.518,51
4,5,baa10y_change_5b,LightGBM,False,0,—,darts_lightgbm,3.45349,51,51,0,0.455,0.200,0.278,0.490,0.488,51
5,5,baa10y_change_5b,Naive,False,0,—,last_value_naive,6.80392,51,51,0,0.423,0.440,0.431,0.431,0.438,51
6,21,baa10y_change_21b,LightGBM,False,0,—,darts_lightgbm,6.66198,51,51,0,0.600,0.360,0.450,0.569,0.648,51
7,21,baa10y_change_21b,LinReg,False,0,—,darts_linreg,6.95295,51,51,0,0.000,0.000,0.000,0.510,0.518,51
8,21,baa10y_change_21b,Naive,False,0,—,last_value_naive,10.07843,51,51,0,0.556,0.600,0.577,0.569,0.594,51


# 2020 Stress

In [12]:
stress_2020_results_df = run_experiment("stress_2020", [naive, linreg, lightgbm])

Running last_value_naive ...
  baa10y_change_1b    mean CRPS = 5.95313  (64 preds)
  baa10y_change_5b    mean CRPS = 23.42188  (64 preds)
  baa10y_change_21b   mean CRPS = 129.17188  (64 preds)
Running darts_linreg ...
  baa10y_change_1b    mean CRPS = 5.23094  (64 preds)
  baa10y_change_5b    mean CRPS = 22.80987  (64 preds)
  baa10y_change_21b   mean CRPS = 73.51658  (64 preds)
Running darts_lightgbm ...
  baa10y_change_1b    mean CRPS = 4.34282  (64 preds)
  baa10y_change_5b    mean CRPS = 21.27208  (64 preds)
  baa10y_change_21b   mean CRPS = 72.63709  (64 preds)


In [13]:
check_direction_accuracy(stress_2020_results_df)

Sanity Check — direction accuracy vs. 0.5 baseline (one-sided binomial p < 0.1):
   model  horizon  dir_accuracy  dir_n_eval
  LinReg        1        0.5625          64
LightGBM        1        0.5938          64
   Naive        1        0.7031          64
  LinReg        5        0.5156          64
LightGBM        5        0.6406          64
   Naive        5        0.7031          64
LightGBM       21        0.4531          64
  LinReg       21        0.4688          64
   Naive       21        0.5781          64

PASS — no model is significantly worse than a coin flip on direction.


True

In [14]:
plotly_display_multivariate_backtest_leaderboard(stress_2020_results_df)


,horizon,target,model,uses_covariates,n_covariates,covariates,predictor_id,mean_crps,n_scores,n_predictions,skipped_origins,dir_precision_up,dir_recall_up,dir_f1_up,dir_accuracy,dir_roc_auc_prob_up,dir_n_eval
0,1,baa10y_change_1b,LightGBM,False,0,—,darts_lightgbm,4.34282,64,64,0,0.562,0.600,0.581,0.594,0.614,64
1,1,baa10y_change_1b,LinReg,False,0,—,darts_linreg,5.23094,64,64,0,0.750,0.100,0.176,0.562,0.568,64
2,1,baa10y_change_1b,Naive,False,0,—,last_value_naive,5.95313,64,64,0,0.690,0.667,0.678,0.703,0.735,64
3,5,baa10y_change_5b,LightGBM,False,0,—,darts_lightgbm,21.27208,64,64,0,0.630,0.567,0.596,0.641,0.724,64
4,5,baa10y_change_5b,LinReg,False,0,—,darts_linreg,22.80987,64,64,0,0.333,0.033,0.061,0.516,0.361,64
5,5,baa10y_change_5b,Naive,False,0,—,last_value_naive,23.42188,64,64,0,0.690,0.667,0.678,0.703,0.721,64
6,21,baa10y_change_21b,LightGBM,False,0,—,darts_lightgbm,72.63709,64,64,0,0.500,0.143,0.222,0.453,0.579,64
7,21,baa10y_change_21b,LinReg,False,0,—,darts_linreg,73.51658,64,64,0,1.000,0.029,0.056,0.469,0.373,64
8,21,baa10y_change_21b,Naive,False,0,—,last_value_naive,129.17188,64,64,0,0.580,0.829,0.682,0.578,0.556,64
